# Chapter 7: Probabilistic Modeling

> Instead of asking "what's the best hyperplane?", probabilistic modeling asks "what distribution most plausibly generated this data?" — and classification falls out for free.

**Type:** Learn + Build &nbsp;|&nbsp; **Language:** Python &nbsp;|&nbsp; **Prerequisites:** Chapter 6 (Linear Models) &nbsp;|&nbsp; **Time:** ~45 minutes
**Source:** *A Course in Machine Learning*, Hal Daumé III — Chapter 7

---

## Learning Objectives

- Define the generative story behind a Naive Bayes classifier, for both continuous (Gaussian) and binary (Bernoulli) features
- Derive relative-frequency estimation as the solution to a constrained maximum-likelihood optimization problem
- Implement Gaussian Naive Bayes and Bernoulli Naive Bayes from scratch and validate them against scikit-learn
- Compare generative (Naive Bayes) and conditional/discriminative (logistic regression) models trained on the same linear decision boundary
- Explain the classic Naive-Bayes-vs-logistic-regression data-size trade-off

## The Problem

Linear models like the perceptron and SVM (Chapter 6) discriminate directly: they never model how the data was produced, only where the boundary between classes should sit.

**Probabilistic modeling** takes a different philosophy: assume the data distribution `D(x, y)` exists, model it explicitly with a parametric family (e.g., Gaussian, Bernoulli), and estimate its parameters by maximum likelihood. If you knew `D` exactly, the **Bayes optimal classifier** — simply predicting `argmax_y D(x, y)` — would be provably optimal. Since you don't know `D`, you estimate it from data, and **Naive Bayes** is the simplest, most practical instance of this idea.

## The Concept

**The Naive Bayes recipe:**

```
Assume: features are independent given the label
      │
      ├──► Estimate p(y) by counting label frequency
      │
      └──► Estimate p(x_d | y) per feature, per class
                  │
                  ▼
        Combine via Bayes' rule at test time
                  │
                  ▼
   Predict argmax_y  p(y) * product of p(x_d | y)
```

### Key Ideas

- **The naive Bayes assumption:** `p(x_1, ..., x_D | y) = product_d p(x_d | y)`. This is almost always false in reality (words in a sentence are correlated!), but it turns an intractable joint distribution into a product of `D` easy one-dimensional estimation problems.
- **Different feature types, different per-feature models:** binary features → Bernoulli distribution (Section 7.3); continuous features → Gaussian distribution (Section 7.5); counts/categories → Multinomial/Discrete distribution.
- **Maximum likelihood estimation reduces to counting.** For Bernoulli features, `theta_{(y),d}` is just the fraction of class-`y` examples where feature `d` is present (Eq 7.21); no gradient descent required.
- **Naive Bayes' decision boundary is linear** (Section 7.4): the log-likelihood-ratio `log p(y=+1|x) - log p(y=-1|x)` reduces algebraically to `w·x + b`, exactly the same functional form as the perceptron, SVM, and logistic regression from Chapter 6 — only the *estimation procedure* differs (counting vs. gradient descent).

## Build It

### Setup

We use the real Wine dataset (continuous features) for Gaussian Naive Bayes, and the real **SMS Spam Collection** dataset (5,574 real text messages, labeled ham/spam) for Bernoulli Naive Bayes and its comparison against logistic regression.

In [1]:
import csv
import urllib.request
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score

RNG = np.random.RandomState(42)

### Step 1: Gaussian Naive Bayes, From Scratch (Sections 7.3 & 7.5)

The generative story: choose a label `y` according to the class prior `theta_y`, then for each feature `d`, draw `x_d` from a Normal distribution `N(mu_{y,d}, sigma^2_{y,d})`.

Fitting is just counting and computing per-class, per-feature means and variances — no iterative optimization. Prediction scores each class by its log-prior plus the sum of per-feature Gaussian log-densities, and returns the class with the highest total.

In [2]:
class GaussianNaiveBayesFromScratch:
    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        self.theta_ = {}
        self.mu_ = {}
        self.var_ = {}
        N = len(y)
        for c in self.classes_:
            Xc = X[y == c]
            self.theta_[c] = len(Xc) / N
            self.mu_[c] = Xc.mean(axis=0)
            self.var_[c] = Xc.var(axis=0) + 1e-9
        return self

    def _log_joint(self, X):
        X = np.asarray(X)
        log_probs = np.zeros((X.shape[0], len(self.classes_)))
        for i, c in enumerate(self.classes_):
            log_prior = np.log(self.theta_[c])
            var = self.var_[c]
            mu = self.mu_[c]
            log_lik = -0.5 * np.sum(np.log(2 * np.pi * var)) \
                      - 0.5 * np.sum(((X - mu) ** 2) / var, axis=1)
            log_probs[:, i] = log_prior + log_lik
        return log_probs

    def predict(self, X):
        log_probs = self._log_joint(X)
        return self.classes_[np.argmax(log_probs, axis=1)]

### Step 2: Bernoulli Naive Bayes for Text (Section 7.3, Eq. 7.18/7.21)

For binary bag-of-words features, `theta_{(y),d} = P(\text{word } d \text{ present} \mid \text{class } y)` is estimated by **Laplace-smoothed relative frequency** — the `alpha` term prevents any word from ever getting an exact probability of 0 or 1, which would otherwise make the log-likelihood blow up to `-infinity` for unseen combinations.

At test time, each document's score is its class log-prior plus the sum of per-word log-probabilities, using `log theta_d` for present words and `log(1 - theta_d)` for absent ones.

In [3]:
class BernoulliNaiveBayesFromScratch:
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, y):
        X = np.asarray(X.todense()) if hasattr(X, "todense") else np.asarray(X)
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        N, D = X.shape
        self.log_prior_ = {}
        self.log_theta_ = {}
        self.log_one_minus_theta_ = {}
        for c in self.classes_:
            Xc = X[y == c]
            self.log_prior_[c] = np.log(len(Xc) / N)
            counts = Xc.sum(axis=0)
            theta = (counts + self.alpha) / (len(Xc) + 2 * self.alpha)
            self.log_theta_[c] = np.log(theta)
            self.log_one_minus_theta_[c] = np.log(1 - theta)
        return self

    def predict(self, X):
        X = np.asarray(X.todense()) if hasattr(X, "todense") else np.asarray(X)
        scores = np.zeros((X.shape[0], len(self.classes_)))
        for i, c in enumerate(self.classes_):
            scores[:, i] = (
                self.log_prior_[c]
                + X @ self.log_theta_[c]
                + (1 - X) @ self.log_one_minus_theta_[c]
            )
        return self.classes_[np.argmax(scores, axis=1)]

## Use It — Real Data

### Experiment A: Gaussian Naive Bayes on the Wine Dataset vs. `sklearn.naive_bayes.GaussianNB`

We fit our from-scratch Gaussian Naive Bayes on the (continuous-featured) Wine dataset and compare it against scikit-learn's reference `GaussianNB`, checking both accuracy and prediction agreement.

In [4]:
wine = load_wine()
Xw, yw = wine.data, wine.target
print(f"Dataset shape: {Xw.shape[0]} examples, {Xw.shape[1]} features, {len(set(yw))} classes")

Xw_train, Xw_test, yw_train, yw_test = train_test_split(
    Xw, yw, test_size=0.3, random_state=42, stratify=yw
)

gnb_scratch = GaussianNaiveBayesFromScratch().fit(Xw_train, yw_train)
pred_scratch = gnb_scratch.predict(Xw_test)
acc_scratch = accuracy_score(yw_test, pred_scratch)

sk_gnb = GaussianNB().fit(Xw_train, yw_train)
pred_sklearn = sk_gnb.predict(Xw_test)
acc_sklearn = accuracy_score(yw_test, pred_sklearn)

agreement = np.mean(pred_scratch == pred_sklearn)
print(f"From-scratch Gaussian NB test accuracy : {acc_scratch:.4f}")
print(f"sklearn GaussianNB       test accuracy : {acc_sklearn:.4f}")
print(f"Prediction agreement rate              : {agreement:.4f}")

Dataset shape: 178 examples, 13 features, 3 classes
From-scratch Gaussian NB test accuracy : 1.0000
sklearn GaussianNB       test accuracy : 1.0000
Prediction agreement rate              : 1.0000


### Experiment B: Bernoulli Naive Bayes for Text (SMS Spam Collection)

Now a real text-classification task. We download the public **SMS Spam Collection** dataset (5,574 real text messages labeled `ham` or `spam`), vectorize each message into a binary (word-present/absent) bag-of-words representation, and compare our from-scratch Bernoulli Naive Bayes against scikit-learn's `BernoulliNB`.

In [5]:
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
urllib.request.urlretrieve(url, "sms.tsv")

labels, texts = [], []
with open("sms.tsv", encoding="utf-8") as f:
    reader = csv.reader(f, delimiter="\t")
    for row in reader:
        if len(row) != 2:
            continue
        label, text = row
        labels.append(1 if label == "spam" else 0)
        texts.append(text)
labels = np.array(labels)

texts_train, texts_test, ytr, yte = train_test_split(
    texts, labels, test_size=0.25, random_state=42, stratify=labels
)

vectorizer = CountVectorizer(max_features=2000, binary=True, stop_words="english")
Xtr = vectorizer.fit_transform(texts_train)
Xte = vectorizer.transform(texts_test)

print(f"Train messages: {Xtr.shape[0]}, Test messages: {Xte.shape[0]}, Vocabulary size: {Xtr.shape[1]}")
print(f"Classes: ham (0) vs spam (1); spam rate = {labels.mean():.3f}")

bnb_scratch = BernoulliNaiveBayesFromScratch(alpha=1.0).fit(Xtr, ytr)
pred_bnb_scratch = bnb_scratch.predict(Xte)
acc_bnb_scratch = accuracy_score(yte, pred_bnb_scratch)

sk_bnb = BernoulliNB(alpha=1.0).fit(Xtr, ytr)
pred_bnb_sklearn = sk_bnb.predict(Xte)
acc_bnb_sklearn = accuracy_score(yte, pred_bnb_sklearn)

agreement_bnb = np.mean(pred_bnb_scratch == pred_bnb_sklearn)
print(f"From-scratch Bernoulli NB test accuracy : {acc_bnb_scratch:.4f}")
print(f"sklearn BernoulliNB      test accuracy   : {acc_bnb_sklearn:.4f}")
print(f"Prediction agreement rate                : {agreement_bnb:.4f}")

Train messages: 4179, Test messages: 1393, Vocabulary size: 2000
Classes: ham (0) vs spam (1); spam rate = 0.134


From-scratch Bernoulli NB test accuracy : 0.9842
sklearn BernoulliNB      test accuracy   : 0.9842
Prediction agreement rate                : 1.0000


### Experiment C: Generative (Naive Bayes) vs. Conditional (Logistic Regression)

Section 7.6's point: both Naive Bayes and logistic regression compute `log p(y=+1|x) / p(y=-1|x)` as a **linear** function of `x` (Eq. 7.27/7.28). Naive Bayes gets there by assuming feature independence and estimating each `theta` by counting (**generative**), while logistic regression directly optimizes the conditional log-likelihood via gradient descent (**discriminative/conditional**). We train both on the exact same bag-of-words features to compare them head-to-head.

In [6]:
logreg = LogisticRegression(max_iter=2000).fit(Xtr, ytr)
pred_logreg = logreg.predict(Xte)
acc_logreg = accuracy_score(yte, pred_logreg)

print(f"{'Model':<28} | {'Test accuracy':>13}")
print("-" * 46)
print(f"{'Bernoulli Naive Bayes (ours)':<28} | {acc_bnb_scratch:>13.4f}")
print(f"{'Logistic Regression (sklearn)':<28} | {acc_logreg:>13.4f}")

Model                        | Test accuracy
----------------------------------------------
Bernoulli Naive Bayes (ours) |        0.9842
Logistic Regression (sklearn) |        0.9770


**Reading the result:** Naive Bayes is much faster to train (closed-form counting) but can be slightly less accurate when its independence assumption is violated — which is almost always the case for natural language, since words in a sentence are correlated.

### Experiment D: Naive Bayes vs. Logistic Regression as Training Size Grows

A classic result in the generative-vs-discriminative literature (the Ng & Jordan intuition): Naive Bayes tends to converge faster with less data, since its strong independence assumption acts as inductive bias, while logistic regression's less constrained hypothesis space needs more data before it catches up — and can eventually overtake Naive Bayes once the independence assumption starts to hurt more than it helps. We test this by training both models on increasingly large random subsets of the training data.

In [7]:
print(f"{'# train docs':>12} | {'NB test acc':>11} | {'LogReg test acc':>15}")
print("-" * 46)
n_total = Xtr.shape[0]
for frac in [0.02, 0.05, 0.1, 0.25, 0.5, 1.0]:
    n_sub = max(10, int(n_total * frac))
    idx = RNG.choice(n_total, size=n_sub, replace=False)
    X_sub, y_sub = Xtr[idx], ytr[idx]

    nb_sub = BernoulliNaiveBayesFromScratch(alpha=1.0).fit(X_sub, y_sub)
    acc_nb_sub = accuracy_score(yte, nb_sub.predict(Xte))

    lr_sub = LogisticRegression(max_iter=2000).fit(X_sub, y_sub)
    acc_lr_sub = accuracy_score(yte, lr_sub.predict(Xte))

    print(f"{n_sub:>12} | {acc_nb_sub:>11.4f} | {acc_lr_sub:>15.4f}")

# train docs | NB test acc | LogReg test acc
----------------------------------------------
          83 |      0.8658 |          0.8729
         208 |      0.8658 |          0.9060


         417 |      0.8787 |          0.9268


        1044 |      0.9541 |          0.9627
        2089 |      0.9756 |          0.9691


        4179 |      0.9842 |          0.9770


**Reading the table:** with very few examples, both models struggle, but Naive Bayes' counting-based estimates tend to stabilize faster than logistic regression's gradient-based search. On this particular spam-detection task the word-independence assumption happens to be a fairly good match for the data (individual "spammy" words really are strong, largely independent signals), so Naive Bayes can remain competitive with — and sometimes even edge out — logistic regression even at full data size. This won't hold on every dataset: the classic result is that logistic regression tends to catch up or overtake Naive Bayes as data grows, whenever the independence assumption is more badly violated than it is here.

## Use It

| API / Function | When to use it |
|---|---|
| `GaussianNaiveBayesFromScratch()` | Continuous features, want an extremely fast, closed-form generative baseline |
| `BernoulliNaiveBayesFromScratch(alpha)` | Binary/bag-of-words text features; `alpha` controls Laplace smoothing strength |
| `sklearn.naive_bayes.GaussianNB` / `BernoulliNB` | Production use; handles sparse matrices and more numerical edge cases |
| `sklearn.linear_model.LogisticRegression` | When you suspect the independence assumption is badly violated, or want a discriminatively-trained linear boundary instead |

## Exercises

1. Add Laplace smoothing (`alpha`) as a tunable hyperparameter to `GaussianNaiveBayesFromScratch`'s variance estimate (i.e., add `alpha` to `var_` directly) and see how it affects accuracy on classes with very few training examples.
2. Implement Multinomial Naive Bayes (using word *counts* rather than presence/absence) and compare it with the Bernoulli version on the SMS dataset.
3. Extract the top 10 words with the highest `log_theta[spam] - log_theta[ham]` and check whether they match your intuition about what makes a text message "spammy."


## Key Terms

| Term | Common Assumption | Precise Meaning |
|---|---|---|
| **Naive Bayes** | "A weak baseline you outgrow immediately" | A generative classifier whose only approximation is feature independence given the label; often extremely competitive on high-dimensional, sparse data like text |
| **Generative Model** | "Models that generate images or text" | Any model that specifies `p(x, y)` (or `p(x｜y)` and `p(y)`) rather than directly modeling `p(y｜x)`; "generative" refers to being able to sample fictitious data, not to any particular data type |
| **Maximum Likelihood Estimation** | "A complicated statistical procedure" | Choosing parameters that maximize the probability of the observed training data; for Naive Bayes this reduces to simple counting |
| **Bayes Optimal Classifier** | "The best classifier we could ever build" | The theoretical classifier `argmax_y D(x,y)` that is optimal *if* you knew the true data distribution `D` exactly — a benchmark, not a practical algorithm, since `D` is always unknown |

## Summary

- Probabilistic modeling estimates the data distribution `D(x, y)` explicitly, rather than discriminating directly like Chapter 6's linear models
- **Naive Bayes** assumes features are independent given the label, turning an intractable joint estimation problem into `D` simple one-dimensional ones
- Fitting Naive Bayes reduces to **counting** (relative frequencies, with Laplace smoothing), not iterative optimization
- Naive Bayes' decision boundary is **linear**, just like the perceptron, SVM, and logistic regression — only the estimation procedure differs
- Naive Bayes tends to need **less data** to reach a stable estimate than logistic regression, though logistic regression can eventually overtake it when the independence assumption is badly violated

---

**Next:** Chapter 8 — Neural Networks